In [6]:
import os
from pathlib import Path
import json
import numpy as np
import open3d as o3d
from joblib import Parallel, delayed
from tqdm import tqdm

# --- CONFIG ---
INPUT_ROOT = Path(r"D:\CVPR_Data\Matched_SOW_ID_Visit_Frames")
OUTPUT_ROOT = Path(r"D:\CVPR_Data\Matched_SOW_ID_Visit_PLY")
CALIB_DIR = Path(r"C:\Users\spaudel\OneDrive - University of Arkansas System Division of Agriculture\Desktop\Projects_shivapdl\ID_Project\Depth_Camera_Calibration")

CALIB_MAP = {
    "Camera1": str(CALIB_DIR / "First_camera.json"),
    "Camera2": str(CALIB_DIR / "Second_Camera.json"),
    "Camera3": str(CALIB_DIR / "Third_camera.json"),
}

# --- PROCESSOR ---
def process_one(depth_file, debug=False):
    try:
        depth_file = Path(depth_file)
        cam_key = next((c for c in ["Camera1", "Camera2", "Camera3"] if depth_file.name.startswith(c)), None)
        if not cam_key: return None

        # Load Intrinsic
        with open(CALIB_MAP[cam_key], "r") as f:
            calib = json.load(f)
        intrinsic = o3d.camera.PinholeCameraIntrinsic(
            int(calib["rectified.1.width"]), int(calib["rectified.1.height"]),
            float(calib["rectified.1.fx"]), float(calib["rectified.1.fy"]),
            float(calib["rectified.1.ppx"]), float(calib["rectified.1.ppy"])
        )

        # 1. Load 
        depth_raw = o3d.io.read_image(str(depth_file))
        pcd = o3d.geometry.PointCloud.create_from_depth_image(
            depth_raw, intrinsic, depth_scale=1000.0, depth_trunc=1.5, stride=1
        )
        if not pcd.has_points(): return None

        # 2. Z-Filter (Restore original 0.1 - 0.9m)
        pts = np.asarray(pcd.points)
        mask = (pts[:, 2] > 0.1) & (pts[:, 2] < 0.9)
        pcd = pcd.select_by_index(np.where(mask)[0])
        if not pcd.has_points(): return None

        # 3. Voxel Downsample
        pcd = pcd.voxel_down_sample(voxel_size=0.005)

        # 4. FIXED DBSCAN 
        # Increased eps to 0.04 (4cm) to bridge gaps between points on the sow
        labels = np.array(pcd.cluster_dbscan(eps=0.04, min_points=100, print_progress=False))
        
        if labels.size > 0 and labels.max() >= 0:
            counts = np.bincount(labels[labels >= 0])
            big_cluster = np.argmax(counts)
            pcd = pcd.select_by_index(np.where(labels == big_cluster)[0])
        else:
            return None

        # 5. Restore CVPR Paper Threshold (6,000 points)
        if len(pcd.points) < 6000:
            if debug: print(f"[{depth_file.name}] Filtered: Only {len(pcd.points)} pts")
            return None

        # 6. Save
        rel = depth_file.relative_to(INPUT_ROOT)
        out = OUTPUT_ROOT / rel.with_suffix(".ply")
        out.parent.mkdir(parents=True, exist_ok=True)
        o3d.io.write_point_cloud(str(out), pcd, write_ascii=False)
        return str(out)

    except Exception:
        return None

if __name__ == "__main__":
    files = []
    for day_dir in INPUT_ROOT.iterdir():
        if day_dir.is_dir():
            day_files = [Path(r) / f for r, d, fs in os.walk(day_dir) for f in fs if f.endswith("_depth.png")]
            files.extend(day_files)
    
    print(f"Total files: {len(files)}")

    # Parallel processing with threading to avoid Pickle errors
    results = Parallel(n_jobs=12, backend="threading")(
        delayed(process_one)(f) for f in tqdm(files, desc="Processing Days")
    )

    valid_count = len([r for r in results if r is not None])
    print(f"\nDONE. Saved {valid_count} / {len(files)} files.")

  0%|          | 23/95771 [03:20<231:54:30,  8.72s/it]


Total files: 95771


Processing Days: 100%|██████████| 95771/95771 [5:05:00<00:00,  5.23it/s]   



DONE. Saved 29856 / 95771 files.
